# 1. Set up GPU, Import Libraries

In [ ]:
import sys
#sys.path.insert(0,'...Path where the corresponding environment is...')

In [ ]:
import os
import numpy as np
from tqdm import tqdm
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow.keras as keras

from keras.layers import Input, Dense, Reshape, Flatten, Dropout, BatchNormalization, Activation, ZeroPadding2D, LeakyReLU, UpSampling2D, Conv2D
from keras.models import Sequential, Model
from keras.optimizers import Adam
from keras.preprocessing import image
from keras.preprocessing.image import img_to_array, load_img, array_to_img

In [ ]:
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

# 2. Loading Data, disease by disease

In [ ]:
disease_class = {'Atelectasis': [],
                 'Cardiomegaly': [],
                 'Effusion': [],
                 'Infiltration': [],
                 'Mass': [],
                 'Nodule': [],
                 'Pneumonia': [],
                 'Pneumothorax': [],
                 'Consolidation': [],
                 'Edema': [],
                 'Emphysema': [],
                 'Fibrosis': [],
                 'Pleural_Thickening': [],
                 'Hernia': [],
                 'No Finding':[]}

disease_rev = {v: k for k, v in disease_class.items()}

In [ ]:
#import labels of the images
data_ref = pd.read_csv("Data_Entry.csv")
pd.options.mode.chained_assignment = None        #ignore the SettingWithCopyWarning


for i in tqdm(range(len(data_ref))):
    #only record single-labelled images
    
    if "|" not in data_ref['Finding Labels'][i]:
        disease_img[data_ref['Finding Labels'][i]].append(data_ref['Image Index'][i])

In [ ]:
for item in disease_class.keys():
    print(item, end=': ')
    print(len(disease_class[item]))

In [ ]:
# training data is loaded by label, in order to train the GAN model a label by label
train_image = []

#disease = 'Hernia'
#disease = 'Pneumonia'
#disease = 'Edema'
#disease = 'Fibrosis'
#disease = 'Emphysema'
#disease = 'Pleural_Thickening'
#disease = 'Consolidation'
#disease = 'Mass'
#disease = 'Pneumothorax'
#disease = 'Cardiomegaly'
#disease = 'Nodule'
#disease = 'Effusion'
#disease = 'Atelectasis'
#disease = 'Infiltration'
disease = 'No Finding'
image_name = disease_class[disease]#[2000:]

for i in tqdm(range(len(image_name))):
    #an image from original dataset

    img = image.load_img('images/'+image_name[i],target_size=(112,112,3))
    img = image.img_to_array(img)
    img = img/255.
    train_image.append(img)

X_train = np.array(train_image)

# 3. Build DCGAN

In [ ]:
def _check_trainable_weights_consistency(self):
    return
Model._check_trainable_weights_consistency = _check_trainable_weights_consistency

In [ ]:
class DCGAN():
    def __init__(self):
        self.img_rows = 112
        self.img_cols = 112
        self.channels = 3
        self.img_shape = (self.img_rows, self.img_cols, self.channels)
        self.latent_dim = 100
        self.history = []

        optimizer = Adam(0.0002, 0.5)

        # Build and compile the discriminator
        self.discriminator = self.build_discriminator()
        self.discriminator.compile(loss='binary_crossentropy',
                           optimizer=optimizer,
                           metrics=['accuracy'])
        
        # Build the generator
        self.generator = self.build_generator()
        
        # The generator takes noise as input and generates imgs
        z = Input(shape=(self.latent_dim,))
        img = self.generator(z)

        # For the combined model we will only train the generator
        self.discriminator.trainable = False

        # The discriminator takes generated images as input and determines validity
        validity = self.discriminator(img)

        # The combined model  (stacked generator and discriminator)
        # Trains the generator to fool the discriminator
        self.combined = Model(z, validity)
        self.combined.compile(loss='binary_crossentropy', optimizer=optimizer)

    def build_generator(self):

        model = Sequential()

        model.add(Dense((256*14*14), activation='relu', input_dim=self.latent_dim))
        model.add(Reshape((14,14,256)))
        model.add(UpSampling2D())
        
        model.add(Conv2D(256, kernel_size=3, padding='same'))
        model.add(BatchNormalization(momentum=0.8))
        model.add(Activation('relu'))
        model.add(UpSampling2D())
        
        model.add(Conv2D(128, kernel_size=3, padding='same'))
        model.add(BatchNormalization(momentum=0.8))
        model.add(Activation('relu'))
        model.add(UpSampling2D())
        
        model.add(Conv2D(64, kernel_size=3, padding='same'))
        model.add(BatchNormalization(momentum=0.8))
        model.add(Activation('relu'))
        
        model.add(Conv2D(self.channels, kernel_size=3, padding='same'))
        model.add(Activation('tanh'))

        model.summary()

        noise = Input(shape=(self.latent_dim,))
        img = model(noise)

        return Model(noise, img)

    def build_discriminator(self):

        model = Sequential()
        
        model.add(Conv2D(32, kernel_size=3, strides=2, input_shape=self.img_shape, padding='same'))
        model.add(LeakyReLU(alpha=0.2))
        model.add(Dropout(0.25))
        
        model.add(Conv2D(64, kernel_size=3, strides=2, padding='same'))
        model.add(ZeroPadding2D(padding=((0,1),(0,1))))
        model.add(BatchNormalization(momentum=0.8))
        model.add(LeakyReLU(alpha=0.2))
        model.add(Dropout(0.25))
        
        model.add(Conv2D(128, kernel_size=3, strides=2, padding='same'))
        model.add(BatchNormalization(momentum=0.8))
        model.add(LeakyReLU(alpha=0.2))
        model.add(Dropout(0.25))
        
        model.add(Conv2D(256, kernel_size=3, strides=2, padding='same'))
        model.add(BatchNormalization(momentum=0.8))
        model.add(LeakyReLU(alpha=0.2))
        model.add(Dropout(0.25))
        
        model.add(Flatten())
        model.add(Dense(1, activation='sigmoid'))
        model.summary()

        img = Input(shape=self.img_shape)
        validity = model(img)

        return Model(img, validity)

    def train(self, epochs, batch_size):
        
        # Adversarial ground truths
        valid = np.ones((batch_size, 1))
        fake = np.zeros((batch_size, 1))

        for epoch in range(epochs):
            
            # ---------------------
            #  Train Discriminator
            # ---------------------
            idx = np.random.randint(0, X_train.shape[0], batch_size)
            images = X_train[idx]
            
            noise = np.random.normal(0, 1, (batch_size, self.latent_dim))

            # Generate a batch of new images
            gen_imgs = self.generator.predict(noise)

            # Train the discriminator
            d_loss_real = self.discriminator.train_on_batch(images, valid)
            d_loss_fake = self.discriminator.train_on_batch(gen_imgs, fake)
            d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

            # ---------------------
            #  Train Generator
            # ---------------------
            noise = np.random.normal(0, 1, (batch_size, self.latent_dim))

            # Train the generator (to have the discriminator label samples as valid)
            g_loss = self.combined.train_on_batch(noise, valid)

            # Plot the progress
            print ("%d [D loss: %f] [G loss: %f]" % (epoch, d_loss[0], g_loss))
                
            self.history.append({"D": d_loss[0], "G": g_loss})
    
    def _predict_noise(self, size):
        noise = np.random.normal(0, 1, (size, self.latent_dim))
        return self.generator.predict(noise)
        
    def gen(self, size):
        generated = self._predict_noise(size)
        generated = 0.5 * generated + 0.5
        return generated

    def sample_images(self, epoch):
        r, c = 1, 1
        noise = np.random.normal(0, 1, (r*c, self.latent_dim))
        gen_img = self.generator.predict(noise)

        # Rescale images 0 - 1
        gen_img = 0.5 * gen_img + 0.5

        fig, ax = plt.subplots()

        ax.imshow(gen_img[0,:,:,0], cmap='gray')
        ax.axis('off')
        fig.savefig("GAN/Sample"+"_%d_" % epoch+disease, bbox_inches='tight')
        plt.close()
    
    def plot_loss(self):
        hist = pd.DataFrame(self.history)
        plt.figure(figsize=(20,5))
        for colnm in hist.columns:
            plt.plot(hist[colnm],label=colnm)
        plt.legend()
        plt.ylabel("loss")
        plt.xlabel("epochs")
        plt.show()
        

# 4. Train with data (formal)

In [ ]:
if __name__ == '__main__':
    dcgan = DCGAN()
    dcgan.train(epochs=10000, batch_size=16)


# 5. Training Performance of DCGAN

In [ ]:
dcgan.plot_loss()

# 6. Save Generated Images

In [ ]:
gan_images = dcgan.gen(len(image_name))

In [ ]:
for i in tqdm(range(len(image_name))):
    plt.imsave("GAN/G"+image_name[i], gan_images[i,:,:,0], cmap = 'gray')